In [48]:
import json
import os
from datetime import datetime, timedelta

from langchain_community.tools import DuckDuckGoSearchResults

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai_tools import CSVSearchTool

import yfinance as yf

from IPython.display import Markdown

In [ ]:
os.environ['OPENAI_API_KEY'] = "Openai_API_Key"
llm = LLM(model= "gpt-4o-mini")

In [3]:
csv_wallet_tool = CSVSearchTool(csv='./Wallet/wallet.csv')

In [4]:
customerManager = Agent(
    role="Customer Stocks Manager",
    goal="Get the customer question about the stock {ticket} and search the customer wallet CSV file for the stocks.",
    backstory="""You're the customer manager of the customer invetiments wallet.
    You are the client first contact and you provide the information the other analystis with the necessary stock ticket and wallet information.
    """,
    verbose=True,
    llm=llm,
    max_iter=5,
    tools=[csv_wallet_tool],
    allow_delegation=False,
    memory=True,
)

In [5]:
getCustomerWallet = Task(
    description="""Use the customer question and find the {ticket} in the CSV File.
    Provide if the stock is in the customer wallet and if it is, provide with the mean price he paid
    and the total numbers of sotcks onwed.
    """,
    expected_output="If the cutomer owns the stocks, provide the mean price paid and the total stock numbers.",
    agent=customerManager
)

In [6]:
stocketPriceAnalyst = Agent(
    role="Senior Stock Price Analyst",
    goal="Find the {ticket} stock price and analyses its trends. Compare with the price that the customer paid",
    backstory="""You're a highly expirenced analyzing the price of specific sotkcs
    make predictions about its future price.""",
    verbose=True,
    llm=llm,
    max_iter=5,
    allow_delegation=False,
    memory=True,
)

In [30]:
# Criando Yahoo Finance
@tool("Yahoo Finance Tool")
def fetch_stock_price(ticket: str):
    """Fetches stocks prices from the last year about a specific company from the Yahoo Finance API."""
    end_date = datetime.today()
    start_date = end_date - timedelta(days=365)
    stock = yf.download(ticket, start=start_date.strftime("%Y-%m-%d"), end=end_date.strftime("%Y-%m-%d"))
    return stock

In [31]:
getStockPrice = Task(
    description="Analyze the stock {ticket} price history and create a trend analyses of up, down or sideways",
    expected_output="""Specify the current trend stocks price - Up, down or sideway.
    eg. stock=AAPL, pirce UP.
    """,
    tools=[fetch_stock_price],
    agent=stocketPriceAnalyst
)

In [32]:
newAnalyst = Agent(
    role="News Analyst",
    goal="""Create a short summary of the market news related to the stock {ticket} company.
    Provide a market Fear & Greed Index score about the company.
    For each requested stock asset, specify a number between 0 and 100, where 0 is extremely fear and 100 is extreme greed.
    """,
    backstory="""You're highly experienced in analyzing market trends and news for more then 10 years.
    You're also a master level analyst in the human psychology.
    
    You understand the news, their title and information, but you look at those with a healthy dose of skeptcism.
    You consider the source of the news articles.""",
    verbose=True,
    llm=llm,
    max_iter=5,
    allow_delegation=False,
    memory=True,
)

In [41]:
ddg_search = DuckDuckGoSearchResults(backend="news", num_results=10)

@tool("Search News Tool")
def search_tool(query: str) -> str:
    """Use this tool to search news about a specific stock ticker."""
    return ddg_search.run(query)

In [34]:
getNews = Task(
    description=f"""Use the search tool to search news about the stock ticket.
    The current date is {datetime.now()}
    Compose the results into a helpfull report.
    """,
    expected_output="""A summary of the overall market one sentence summary for the requested asset.
    Include the fear/greed score based on the news. Use format:
    <STOCK TICKET>
    <SUMMARY BASED ON NEWS>
    <FEAR/GREED SCORE>
    """,
    tools=[search_tool],
    agent=newAnalyst,
    
)

In [42]:
stockRecommender = Agent(
    role="Chief Stock Analyst",
    goal="""Get the data from the customer currenly stocks, the provided input of stock price trends and
    the stock news to provide a recommendation: Buy, Sell, or Hold the stock.""",
    backstory="""You're the leader of the stock analyst team. You have a great performance in the past 20 years in stock recommendation.
    With all your team informations, you are able to provide the best recommendation for the customer to achieve
    the maximum value creation.""",
    verbose=True,
    llm=llm,
    max_iter=5,
    allow_delegation=True,
    memory=True,
)

In [43]:
recommendStock = Task(
    description="""Use the stock price trend, the stock news report and the customers stock mean price of the {ticket}
    to provide a recommendatio: Buy, Sell or Hold. If the previous reports are not well provided you can delegate back
    to the specific analyst to work again the their task.
    """,
    expected_output="""A brief recommendation paragraph with the summary of the reasons for the recommendation and the
    recommendation it self in one of the three possible outputs: Buy, Sell or Hold. Use the format:
    <SUMMARY OF REASONS>
    <RECOMMENDATION>
    """,
    agent=stockRecommender,
    context=[getCustomerWallet, getStockPrice, getNews],
)

In [44]:
copyWrite = Agent(
    role="Stock Content Writer",
    goal="""Write an insghtfull compelling and informative 6 paragraph long newletter
    based on the stock price report, the news report and the recommendation report.
    """,
    backstory="""You're a unbeliveble copy writer that understand complex financel conecepts
    and explain for a dummie audience.
    You create complelling stories and narratives that resonate with the audience.
    """,
    verbose=True,
    llm=llm,
    max_iter=5,
    allow_delegation=False,
    memory=True,
)

In [45]:
writeNewsLetter = Task(
    description="""Use the stock price trend, the stock news report and the stock recommendation to write insghtfull compelling and informative 6 paragraph long newletter.
    Focus on the stock price trend, the news and fear/greed score and the summary for the recommendation.
    Include the recommendation in the newsletter.    
    """,
    expected_output="""An eloquent 6 paragraph newsletter formated as Markdown in an easy readable manner. It should contain:
    - Introduct - set the overal picture
    - Main part - provides the meat of the analysis including stock price trend, the news and fear/greed score and the summary for the recommendation.
    - 3 bullets of the main summary reasons of the recommendation
    - Recommendation Summary
    - Recommendation it self
    """,
    agent=copyWrite,
    context=[getStockPrice, getNews, recommendStock]
)

In [46]:
crew = Crew(
    agents=[customerManager, stocketPriceAnalyst, newAnalyst, stockRecommender, copyWrite],
    tasks=[getCustomerWallet, getStockPrice, getNews, recommendStock, writeNewsLetter],
    verbose=True,
    process=Process.hierarchical,
    full_output=True,
    share_crew=False,
    manager_llm=llm,
    max_iter=15
)

In [47]:
result = crew.kickoff(inputs={"ticket": "Give your thoughts about the Amazon Stocks"})

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.2                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f33812a1-0752-4f08-ae53-48d639afccf3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the customer question and find the Give your thoughts about the Amazon Stocks in the CSV File.       │
│      Provide if the stock is in the customer wallet and if it is, provide with the mean price he paid           │
│      and the total numbers of sotcks onwed.                                                                     │
│                                                                                                                 │
│  ID: 6a1746fa-aded-4215-b2d0-a5c151848c7f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Use the customer question and find the Give your thoughts about the Amazon Stocks in the CSV File.       │
│      Provide if the stock is in the customer wallet and if it is, provide with the mean price he paid           │
│      and the total numbers of sotcks onwed.                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_csvs_content                                                                                    │
│  Args: {'search_query': 'Amazon Stocks'}                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Use the stock price trend, the stock news report and the stock recommendation to write insghtfull        │
│  compelling and informative 6 paragraph long newletter.                                                         │
│      Focus on the stock price trend, the news and fear/greed score and the summary for the recommendation.      │
│      Include the recommendation in the newsletter.                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Amazon Stocks Newsletter                                                                                     │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│                                                                                                                 │
│  In the ever-evolving landscape of technology stocks, Amazon.com, Inc. (AMZN) continues to grab the attention   │
│  of investors and analysts alike. As one of the pillars of the tech industry, the stock's performance is        │
│  dissected weekly, providing insights into market trends, investor sentiment, and overall economic health.      │
│  Today, we delve into the latest developments surrounding Amazon and what they mean for your investment         │
│  portfolio.                                                                                                     │
│                                                                                                                 │
│  ## Main Analysis                                                                                               │
│                                                                                                                 │
│  Currently, Amazon's stock price trend is pointed upwards, indicating strength in the market. This positive     │
│  trend is reflective of a broader bullish sentiment surrounding tech stocks in general. Market analysts have    │
│  noted a range of factors fueling this upward movement, not the least of which are recent earnings reports      │
│  that have exceeded expectations. Despite a minor dip following these reports due to higher capital             │
│  expenditures and soft guidance for the second quarter, the overall outlook remains favorable.                  │
│                                                                                                                 │
│  Recent news about Amazon has only bolstered this sentiment. Analysts have categorized it as one of the best    │
│  American tech stocks to buy, which serves as an indicator of strength and stability in a volatile market       │
│  environment. The fear/greed score currently rests at 70, signaling a Greedy market sentiment. This score       │
│  reflects an optimistic investor outlook, promoting the idea that Amazon is well-positioned to capitalize on    │
│  future growth opportunities in emerging technologies and e-commerce.                                           │
│                                                                                                                 │
│  Investors should indeed be cautious as they weigh their options, given the highlighted concerns over capital   │
│  spending. These investments are crucial for ongoing innovations and expansions, yet they also carry risks      │
│  that can weigh heavily on short-term performance. With strong potential for long-term growth, the balanced     │
│  approach becomes key in managing these risks effectively while benefiting from the stock's upward trajectory.  │
│                                                                                                                 │
│  ### Summary of Recommendation Reasons                 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Use the stock price trend, the stock news report and the stock recommendation to write insghtfull        │
│  compelling and informative 6 paragraph long newletter.                                                         │
│      Focus on the stock price trend, the news and fear/greed score and the summary for the recommendation.      │
│      Include the recommendation in the newsletter.                                                              │
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f33812a1-0752-4f08-ae53-48d639afccf3                                                                       │
│  Final Output: # Amazon Stocks Newsletter                                                                       │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│                                                                                                                 │
│  In the ever-evolving landscape of technology stocks, Amazon.com, Inc. (AMZN) continues to grab the attention   │
│  of investors and analysts alike. As one of the pillars of the tech industry, the stock's performance is        │
│  dissected weekly, providing insights into market trends, investor sentiment, and overall economic health.      │
│  Today, we delve into the latest developments surrounding Amazon and what they mean for your investment         │
│  portfolio.                                                                                                     │
│                                                                                                                 │
│  ## Main Analysis                                                                                               │
│                                                                                                                 │
│  Currently, Amazon's stock price trend is pointed upwards, indicating strength in the market. This positive     │
│  trend is reflective of a broader bullish sentiment surrounding tech stocks in general. Market analysts have    │
│  noted a range of factors fueling this upward movement, not the least of which are recent earnings reports      │
│  that have exceeded expectations. Despite a minor dip following these reports due to higher capital             │
│  expenditures and soft guidance for the second quarter, the overall outlook remains favorable.                  │
│                                                                                                                 │
│  Recent news about Amazon has only bolstered this sentiment. Analysts have categorized it as one of the best    │
│  American tech stocks to buy, which serves as an indicator of strength and stability in a volatile market       │
│  environment. The fear/greed score currently rests at 70, signaling a Greedy market sentiment. This score       │
│  reflects an optimistic investor outlook, promoting the idea that Amazon is well-positioned to capitalize on    │
│  future growth opportunities in emerging technologies and e-commerce.                                           │
│                                                                                                                 │
│  Investors should indeed be cautious as they weigh their options, given the highlighted concerns over capital   │
│  spending. These investments are crucial for ongoing innovations and expansions, yet they also carry risks      │
│  that can weigh heavily on short-term performance. With strong potential for long-term growth, the balanced     │
│  approach becomes key in managing these risks effectively while benefiting from the stock's upward trajectory.  │
│                                                                                                                 │
│  ### Summary of Recommendation Reasons                



┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└─────────────────────────────────────

In [49]:
result.raw

"# Amazon Stocks Newsletter\n\n## Introduction\n\nIn the ever-evolving landscape of technology stocks, Amazon.com, Inc. (AMZN) continues to grab the attention of investors and analysts alike. As one of the pillars of the tech industry, the stock's performance is dissected weekly, providing insights into market trends, investor sentiment, and overall economic health. Today, we delve into the latest developments surrounding Amazon and what they mean for your investment portfolio.\n\n## Main Analysis\n\nCurrently, Amazon's stock price trend is pointed upwards, indicating strength in the market. This positive trend is reflective of a broader bullish sentiment surrounding tech stocks in general. Market analysts have noted a range of factors fueling this upward movement, not the least of which are recent earnings reports that have exceeded expectations. Despite a minor dip following these reports due to higher capital expenditures and soft guidance for the second quarter, the overall outlook

In [50]:
Markdown(result.raw)

# Amazon Stocks Newsletter

## Introduction

In the ever-evolving landscape of technology stocks, Amazon.com, Inc. (AMZN) continues to grab the attention of investors and analysts alike. As one of the pillars of the tech industry, the stock's performance is dissected weekly, providing insights into market trends, investor sentiment, and overall economic health. Today, we delve into the latest developments surrounding Amazon and what they mean for your investment portfolio.

## Main Analysis

Currently, Amazon's stock price trend is pointed upwards, indicating strength in the market. This positive trend is reflective of a broader bullish sentiment surrounding tech stocks in general. Market analysts have noted a range of factors fueling this upward movement, not the least of which are recent earnings reports that have exceeded expectations. Despite a minor dip following these reports due to higher capital expenditures and soft guidance for the second quarter, the overall outlook remains favorable.

Recent news about Amazon has only bolstered this sentiment. Analysts have categorized it as one of the best American tech stocks to buy, which serves as an indicator of strength and stability in a volatile market environment. The fear/greed score currently rests at 70, signaling a Greedy market sentiment. This score reflects an optimistic investor outlook, promoting the idea that Amazon is well-positioned to capitalize on future growth opportunities in emerging technologies and e-commerce.

Investors should indeed be cautious as they weigh their options, given the highlighted concerns over capital spending. These investments are crucial for ongoing innovations and expansions, yet they also carry risks that can weigh heavily on short-term performance. With strong potential for long-term growth, the balanced approach becomes key in managing these risks effectively while benefiting from the stock's upward trajectory.

### Summary of Recommendation Reasons
- **Strong Upward Trend**: The stock is currently trending upwards, positively influenced by recent favorable earnings reports.
- **Positive Analyst Sentiment**: Amazon has been recognized as a top tech stock to consider for investment, reinforcing its strong market position.
- **Cautious Optimism**: A high fear/greed score indicates a greedy market sentiment, suggesting that now might be a good time to hold rather than sell.

## Recommendation Summary

Taking into account the aforementioned factors, the general market trend, and the current investor sentiment, we believe that the best course of action with Amazon stocks is to maintain your investment. Given the upward price trend and the positive market conditions suggesting a potential for future growth, holding onto Amazon stocks presents a strategically sound decision.

**Recommendation**: Hold